# aw_08_b5 — Stage B5: PlayWorld DPO from the B4 policy (Track B, §5.1)

**Parent = B4** (`20260812-235057--b4-playworld-sft-from-p1--s42--72f4f6`,
output sha256:378ba470...). Mirrors A2 exactly (same mining recipe, budget,
DPO profile); the only difference is the parent line (base→B1v2→B4 vs base→A1).
Primary readouts: B5 vs B4 (does DPO still help after two-stage SFT?) and
B5 vs A2 (two-stage vs direct under the same DPO recipe).

**x15 FAILED (2026-08-13, deviation D1)** — B4 is being retrained as B4v2 on the
frozen A1-matched artifact (aw_07 §B4v2). This stage now starts from **B4v2**:
set B4_RUN_ID below to the B4v2 run id once its eval/analysis is accepted.

Cell order: x15 gate → fetch B4 → `a_b5_data` → `b_b5_train` → `c_b5_eval` →
`x09g_run_audit` → `f_b5_analysis`.


**v1.3 note:** x16 showed A1 also trained on an unfrozen build ⇒ remediation is a
paired retrain (A1v2 + B4v2 on the pinned canonical artifact, aw_07). B5 parents on
**B4v2**; the B5-vs-A2 comparison requires **A2v2** (re-mine + DPO from A1v2 under
the A2 recipe) — do not compare B5 against v1 A2.


**2026-08-14:** RQ1 closed (B4v2≫A1v2, 10/10 p=.0001). This stage now runs BOTH
Phase-2 DPO arms in parallel under the identical recipe: **A2v2** (parent a1v2
269d0e) and **B5** (parent b4v2 c56ed2). Primary readouts: B5 vs B4v2 (DPO gain
on two-stage), A2v2 vs A1v2 (DPO gain on direct), B5 vs A2v2 (two-stage vs
direct after DPO — the Phase-2 headline).


**STAGE RECORD (CLOSED 2026-08-14).** Mining: b4v2 1350 pairs (yield .675,
no_passed 533) vs a1v2 893 (yield .447, no_passed 1107) — the two-stage parent
leaves the miner far more usable signal. DPO runs a2v2 ef1e20 (margins .45,
acc .85) / b5 4c45da (margins .07, acc .66 — weak gradient: parent already
strong on its own rollouts). Evals 61d7cd / f77cd8, freeze 3cdcbc30, x09:
b5 trunc/runaway 0.0; a2v2 trunc .993/runaway .91 (A-track pathology persists
through DPO). Findings: (1) **B5 vs B4v2 ~null** — only sig cell adversarial
pass −.037 (p=.047, isolated at α=.05 over 10 cells); DPO adds nothing after
two-stage SFT under this recipe. (2) A2v2 vs A1v2: small sig mean_score gains
(adv +.074, template +.044, comp +.026) but pass_rate gains n.s. except
adversarial +.053 — DPO helps the weaker direct arm modestly. (3) **Phase-2
headline: B5 ≫ A2v2 on all 10 cells (p≤.0002)** — the two-stage advantage
survives identical DPO. §6 Phase-2 champion: B5-vs-B4v2 proxy tie → rule-OOD
tie → GPU-hours → **champion = B4v2** (SFT). B6 (GRPO, aw_09) initializes
from B4v2, yielding a clean online-RL vs offline-DPO comparison from one parent.


In [ ]:
# @title common header
import os

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')

!git clone https://{os.environ["GITHUB_TOKEN"]}@github.com/m97j/axiom-world.git
%cd axiom-world
!pip install -e . -r requirements/colab-g4.lock.txt


In [ ]:
# @title x15 gate — RESULT: CONTENT DIVERGED (2026-08-13). Kept for the record;
# do NOT rerun. B5 proceeds from B4v2 (see aw_07 §B4v2).
# Frozen A1-era artifact vs the aw_07 rebuild. Adjust --path if the artifact
# lives elsewhere in m97j/aw-playworld.
!python scripts/fetch_dataset.py \
  --repo m97j/aw-playworld \
  --path preference_train/v1/playworld_sft.jsonl \
  --output data/frozen/playworld_sft_a1era.jsonl

!python scripts/build_training_data.py \
  --seed 1042 --scenarios-per-family 400 --output-dir data/train

!python scripts/x15_sft_data_diff.py \
  --file-a data/frozen/playworld_sft_a1era.jsonl --label-a a1-era-frozen \
  --file-b data/train/playworld_sft.jsonl --label-b rebuild \
  --out runs/x15_sft_data_diff.json


In [ ]:
# @title fetch parents — B4v2 (treatment) + A1v2 (control)
B4V2_RUN_ID = "20260814-023603--b4v2-playworld-sft-from-p1--s42--c56ed2"
A1V2_RUN_ID = "20260814-022256--a1v2-playworld-sft--s42--269d0e"

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b4 --run-id {B4V2_RUN_ID}
b4v2_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]
out = !python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {A1V2_RUN_ID}
a1v2_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

import json
b4v2_sha = json.load(open(f"runs/{B4V2_RUN_ID}/artifacts/lineage.json"))["output_adapter_sha256"]
a1v2_sha = json.load(open(f"runs/{A1V2_RUN_ID}/artifacts/lineage.json"))["output_adapter_sha256"]
print(b4v2_dir, b4v2_sha); print(a1v2_dir, a1v2_sha)


In [ ]:
# @title a_b5_data — mine pairs from BOTH parents (identical recipe, GPU)
# Prompts: canonical frozen prompt stream (prompt_fingerprint cc2aef0d is stable,
# but fetch the frozen copy anyway per the v1.3 rule).
!python scripts/fetch_dataset.py \
  --repo m97j/aw-playworld --path train/v1/playworld_prompts.jsonl \
  --output data/train/playworld_prompts.jsonl --force
!python scripts/build_eval_suites.py --episodes-per-suite 300

for label, adp in (("b4", b4v2_dir), ("a1", a1v2_dir)):
    !python scripts/mine_playworld_pairs.py \
      --config configs/experiments/eval_playworld.yaml \
      --adapter-dir {adp} \
      --prompt-file data/train/playworld_prompts.jsonl \
      --num-candidates 8 --temperature 0.8 --batch-size 100 \
      --selection-method hybrid_verifier_rank --minimum-margin 0.10 \
      --output data/train/playworld_preference_{label}.jsonl \
      --hf-sync-repo m97j/aw-playworld --hf-path-in-repo preference_train/{label}v2-v1


In [ ]:
# @title b_train — A2v2 then B5 (same DPO recipe, lineage-verified)
!python scripts/run_experiment.py \
  --config configs/experiments/a2_playworld_dpo.yaml \
  --parent-adapter-dir {a1v2_dir} \
  --override lineage.parent_run_id={A1V2_RUN_ID} \
  --override lineage.parent_adapter.sha256={a1v2_sha} \
  --override experiment_name=a2v2-playworld-dpo \
  --override data.source.local_path=data/train/playworld_preference_a1.jsonl \
  --hf-sync-repo m97j/aw-runs-a2

!python scripts/run_experiment.py \
  --config configs/experiments/b5_playworld_dpo.yaml \
  --parent-adapter-dir {b4v2_dir} \
  --override lineage.parent_run_id={B4V2_RUN_ID} \
  --override lineage.parent_adapter.sha256={b4v2_sha} \
  --override data.source.local_path=data/train/playworld_preference_b4.jsonl \
  --hf-sync-repo m97j/aw-runs-b5


In [ ]:
# @title c_eval — both DPO arms on the frozen suites
A2V2_RUN_ID = ""  # <- from b_train
B5_RUN_ID = ""

out = !python scripts/fetch_run.py --repo m97j/aw-runs-a2 --run-id {A2V2_RUN_ID}
a2v2_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]
out = !python scripts/fetch_run.py --repo m97j/aw-runs-b5 --run-id {B5_RUN_ID}
b5_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {a2v2_dir} \
  --max-new-tokens 1024 --batch-size 100 --hf-sync-repo m97j/aw-runs-a2
!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {b5_dir} \
  --max-new-tokens 1024 --batch-size 100 --hf-sync-repo m97j/aw-runs-b5


In [ ]:
# @title x09i_run_audit — both DPO evals (CPU)
A2V2_EVAL = ""  # <- eval run ids from c_eval
B5_EVAL = ""

!python scripts/fetch_run.py --repo m97j/aw-runs-a2 --run-id {A2V2_EVAL} --kind eval
!python scripts/fetch_run.py --repo m97j/aw-runs-b5 --run-id {B5_EVAL} --kind eval
!python scripts/x09_termination_audit.py \
  --run-dirs runs/{A2V2_EVAL} runs/{B5_EVAL} --out runs/x09_run_audit_phase2dpo.json


In [ ]:
# @title f_analysis — Phase-2 comparisons
B4V2_EVAL = "20260814-032546--eval-playworld--s42--7308ee"
A1V2_EVAL = "20260814-025057--eval-playworld--s42--a27857"

!python scripts/fetch_run.py --repo m97j/aw-runs-b4 --run-id {B4V2_EVAL} --kind eval
!python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {A1V2_EVAL} --kind eval

# DPO gain per track
!python scripts/run_analysis.py \
  --run-a runs/{B5_EVAL} --label-a b5-dpo --run-b runs/{B4V2_EVAL} --label-b b4v2-sft \
  --output runs/{B5_EVAL}/analysis_b5_vs_b4v2.json --hf-sync-repo m97j/aw-runs-b5
!python scripts/run_analysis.py \
  --run-a runs/{A2V2_EVAL} --label-a a2v2-dpo --run-b runs/{A1V2_EVAL} --label-b a1v2-sft \
  --output runs/{A2V2_EVAL}/analysis_a2v2_vs_a1v2.json --hf-sync-repo m97j/aw-runs-a2

# Phase-2 headline: two-stage vs direct after identical DPO
!python scripts/run_analysis.py \
  --run-a runs/{B5_EVAL} --label-a b5-two-stage-dpo --run-b runs/{A2V2_EVAL} --label-b a2v2-direct-dpo \
  --output runs/{B5_EVAL}/analysis_b5_vs_a2v2.json --hf-sync-repo m97j/aw-runs-b5


## Stage checklist — CLOSED 2026-08-14
- [x] Mining manifests: b4v2 1350 pairs (.675) vs a1v2 893 (.447) — recorded
- [x] a2v2 ef1e20 / b5 4c45da lineage verified (parents 70c2ecb9 / d4fcacdd)
- [x] Evals 61d7cd / f77cd8; freeze 3cdcbc30; x09: b5 0.0/0.0, a2v2 .993/.91
- [x] B5 vs B4v2: ~null (adversarial pass −.037 p=.047 isolated) — DPO no gain after two-stage
- [x] A2v2 vs A1v2: modest sig mean_score gains, pass_rate mostly n.s.
- [x] **Phase-2 headline: B5 ≫ A2v2, 10/10 p≤.0002**
- [x] §6 Phase-2 champion = **B4v2** (proxy tie → rule-OOD tie → GPU-hours)
- [x] Next: B6 GRPO from B4v2 (aw_09_b6)
